# 01 — Exploratory data analysis

**Machine Learning · M.EIC001 · Group G51**

This notebook covers the **business understanding** recap and the **data understanding** phase of CRISP-DM. It ends with a list of decisions that feed the data preparation notebook.

## 0. Setup

In [6]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn scipy nycflights13
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
DATA_DIR = Path("hockey_development_data") / "hockey_development_data"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Business understanding

**Business goals**

Before the start of each season clubs, fans and media all want to know the teams final standing are, and which teams are more likely to fire a coach. We’ll try to answer these questions with the 100 years of NHL dataset.
So our goals will be:
- Predict the final regular-season ranking of every team.
- Identify which teams are more likely to replace their coach during the season.
- Understand the reasons for what drives both outcomes.

We’ll be working with the following assumptions:
- Predictions are made the day before the first game, so nothing from the target season can be used as an input.
- The target is the conference rank, as agreed with the professor.
- No data from the outside can be used to enrich our prediction model.



**Data mining goals**

| Task | Target | DM task type |
|---|---|---|
| (a) | **Points percentage** of each team in the season (points won / points available), from which the final **conference** rank is derived by sorting within the conference | Regression, then ranking |
| (b) | Teams most likely to have a **mid-season** coach replacement | Binary classification |
| (c) | TBA | TBA |

We predict points percentage and not the rank directly, the rank is relative to the other teams in the conference, so it changes with the number of teams, while points percentage describes one team and is comparable across eras (raw points are not, because games per season and the points system changed). The predictions of all teams in a conference are sorted to obtain the rank, which is what we evaluate. Predicting the rank directly will still be a future possibility.

**Data mining success criteria**

| Task | Metrics | Baseline to beat |
|---|---|---|
| (a) | Spearman correlation between predicted and actual conference rank, and mean absolute rank error. MAE of points percentage for the model fit | Last season's rank (new teams get the mean rank) |
| (b) | Precision, recall, F1 and AUC (PR-AUC as well, because positives are rare) | "No coach change" and the base rate by era |

All models are evaluated on seasons later than the ones they were trained on.

## 2. Collect and inventory the data

The data comes as 22 csv files. In all of them `year` is the year the season started (2005-06 is stored as 2005), so the last season we have is 2009.

In [7]:
def load_tables(data_dir=DATA_DIR):
    return {p.stem: pd.read_csv(p, low_memory=False) for p in sorted(data_dir.glob("*.csv"))}

tables = load_tables()
print(len(tables), "tables loaded")

22 tables loaded


In [8]:
def inventory(tables):
    rows = []
    for name, df in tables.items():
        has_year = "year" in df.columns
        rows.append({
            "table": name,
            "rows": len(df),
            "cols": df.shape[1],
            "first_year": df["year"].min() if has_year else np.nan,
            "last_year": df["year"].max() if has_year else np.nan,
            "pct_missing": df.isna().mean().mean() * 100,
        })
    return pd.DataFrame(rows).set_index("table")

inventory(tables)

,rows,cols,first_year,last_year,pct_missing
table,,,,,
abbrev,58,3,NaN,NaN,0.000
AwardsCoaches,75,5,1930.0,2009.0,20.000
AwardsMisc,120,6,1965.0,2009.0,19.722
AwardsPlayers,2021,6,1917.0,2009.0,22.464
Coaches,1741,14,1909.0,2009.0,20.998
CombinedShutouts,50,8,1929.0,2009.0,0.000
Goalies,4096,23,1909.0,2009.0,30.764
GoaliesSC,31,11,1912.0,1925.0,0.000
GoaliesShootout,346,8,2005.0,2009.0,0.000


Six tables go back to 1909: Teams, TeamSplits, TeamVsTeam, Coaches, Scoring and Goalies. The others only cover a few seasons (TeamsHalf stops in 1920 and the shootout tables start in 2005) and Master has no season because it describes people.

We only care about the NHL, so first we check which leagues are in the data.

In [9]:
league_rows = pd.DataFrame({
    t: tables[t].groupby("lgID").size() for t in ["Teams", "Coaches", "Scoring", "Goalies"]
})
league_span = tables["Teams"].groupby("lgID")["year"].agg(first_year="min", last_year="max")
league_rows.join(league_span).sort_values("first_year")

,Teams,Coaches,Scoring,Goalies,first_year,last_year
lgID,,,,,,
NHA,45,43,586,75,1909,1916
PCHA,41,41,404,47,1911,1923
NHL,1265,1522,40101,3678,1917,2009
WCHL,24,24,321,37,1921,1925
WHA,84,111,2436,259,1972,1978


In [10]:
nhl_years = tables["Teams"].query("lgID == 'NHL'")["year"].drop_duplicates().sort_values()
gaps = sorted(set(range(nhl_years.min(), nhl_years.max() + 1)) - set(nhl_years))

print(f"NHL seasons: {len(nhl_years)} ({nhl_years.min()}–{nhl_years.max()}), missing years in between: {gaps}")
print("NHL team-seasons:", int((tables["Teams"]["lgID"] == "NHL").sum()), "of", len(tables["Teams"]))
print("distinct tmID in the NHL:", tables["Teams"].query("lgID == 'NHL'")["tmID"].nunique(),
      "| distinct franchID:", tables["Teams"].query("lgID == 'NHL'")["franchID"].nunique())

NHL seasons: 92 (1917–2009), missing years in between: [2004]
NHL team-seasons: 1265 of 1459
distinct tmID in the NHL: 55 | distinct franchID: 36


What one row means in each group of tables:

| Group | Tables | One row is |
|---|---|---|
| Team-season | Teams, TeamsHalf, TeamSplits, TeamVsTeam | a team in a season (in half a season for TeamsHalf, against one opponent for TeamVsTeam) |
| Players and goalies | Master, Scoring, Goalies, ScoringShootout, GoaliesShootout | a person for Master, otherwise a player or goalie with a team in a season (`stint` counts the spells when a player is traded) |
| Coaches | Coaches | a coach with a team in a season |
| Playoffs and finals | SeriesPost, TeamsPost, TeamsSC, ScoringSC, GoaliesSC | a playoff series, a team's playoff totals in a season, or a team, skater or goalie in the Stanley Cup final (1912 to 1925 only) |
| Awards and misc | AwardsPlayers, AwardsCoaches, AwardsMisc, HOF, CombinedShutouts, ScoringSup, abbrev | an award given in a year, a Hall of Fame inductee, a shutout shared by two goalies, or a code and its full name for abbrev |

What we learned:
- The files have five leagues: NHA, PCHA, WCHL, WHA and the NHL. The NHL has 1265 of the 1459 team-seasons, so using only the NHL keeps most of the data.
- The NHL has 92 seasons between 1917 and 2009. 2004 is missing because of the lockout, so a "previous season" feature can't just use `year - 1`.
- Master has 7236 player ids and 392 coach ids, and 268 of them are both.
- The NHL has 55 team codes (`tmID`) but only 36 franchises (`franchID`), so a team changes code over time. We look at this in section 3.
- Missing values are high in Master (39%), Scoring (34%), TeamSplits (33%) and Goalies (31%). Section 5 looks at how this depends on the era.

## 3. Keys and relationships

FINDING the key of each table and check the links between them: `tmID` + `year` links teams, coaches and scoring; `playerID` and `coachID` link to `Master`.

In [11]:
def is_key(df, cols):
    # True if cols uniquely identify every row and contain no missing values.
    return (not df[cols].isna().any().any()) and (not df.duplicated(subset=cols).any())

candidate_keys = {
    "Teams": ["year", "tmID"],
    "Coaches": ["coachID", "year", "tmID", "stint"],
    "Scoring": ["playerID", "year", "stint"],
    "Goalies": ["playerID", "year", "stint"],
    "TeamsHalf": ["year", "tmID", "half"],
    "TeamsPost": ["year", "tmID"],
}
pd.Series({t: is_key(tables[t], k) for t, k in candidate_keys.items()}, name="is_key")

Teams        True
Coaches      True
Scoring      True
Goalies      True
TeamsHalf    True
TeamsPost    True
Name: is_key, dtype: bool

## 4. Coverage and league evolution

This section justifies the scope decision (which leagues, which era).


## 5. Data Quality

This section evaluates the quality and completeness of the dataset across the 22 tables. We categorize missingness mechanisms (Missing by Design, MAR, MNAR, MCAR) and define data hygiene decisions to feed the **Data Preparation** phase (`02_data_prep.ipynb`).

---

### 5.1 Missing Value Analysis

Missing values in the dataset fall into four distinct structural categories:

1. **Missing by Design (Domain & Rule-Based Gaps):**
   - **Postseason Metrics:** Columns like `PostGP`, `PostG`, `PostGA`, and `playoff` in `Scoring`, `Goalies`, and `Teams` are missing for teams or players that did not qualify for the playoffs in a given season (~48%–97% missingness).
   - **Rule Changes / Historical Additions:** `SoW` and `SoL` (Shootout wins/losses introduced in 2005) and `OTL` (Overtime Losses introduced in 1999) are absent prior to their official implementation.

2. **Missing at Random (MAR — Era & Record-Keeping Evolution):**
   - Advanced and specialized game metrics in `Scoring` and `Teams` (e.g., `+/-`, `SOG`, `PPG`, `SHG`, `PPC`, `PKC`) exhibit high missing rates (20%–80%) in early NHL eras because the league did not officially track them before the late 1960s/1970s.

3. **Missing Not at Random (MNAR / Unstructured Text):**
   - Free-text annotation columns such as `note` in `AwardsCoaches` (100%), `AwardsMisc` (87.5%), `AwardsPlayers` (93.9%), `SeriesPost` (92.6%), `notes` in `Coaches` (99.4%), and `nameNote` in `Master` (99.8%) contain non-systematic historical comments.

4. **Residual MCAR (< 5% Missingness):**
   - Minor missing values in basic statistics (`GP`, `G`, `A`, `Pts`, `PIM` in `Scoring` and `Goalies`, and `firstName` in `Master`) caused by minor historical record omissions.

---

### 5.2 Integrity & Logic Validation

To ensure downstream feature engineering remains mathematically valid:
- **Game Count Coherence:** We verify whether $\text{GP} = \text{W} + \text{L} + \text{T} + \text{OTL}$.
- **Franchise vs. Team ID:** Team codes (`tmID`) change when teams relocate or rename (e.g., `WIN` to `PHX`), whereas `franchID` tracks the continuous franchise history.
- **Mid-Season Stints:** Players/coaches traded mid-season have multiple rows per year (`stint = 1, 2, ...`). These must be aggregated before merging with team-level data to avoid duplicate records and data leakage.

---

### 5.3 Data Quality Decision Matrix

The table below summarizes the data hygiene actions to be executed during the **Data Preparation** phase:

| Data Quality Issue | Root Cause / Mechanism | Decision / Action for Data Prep |
| :--- | :--- | :--- |
| **Text / Comment Columns** (`note`, `notes`, `nameNote`) | MNAR (Unstructured comments) | **Drop Columns** (Remove during preprocessing to eliminate noise). |
| **Missing Rule Metrics** (`SoL`, `SoW`, `OTL`) | Missing by Design (Pre-1999 / Pre-2005) | **Impute `0`** to allow correct calculation of historical Points Percentage. |
| **Postseason Metrics** (`PostGP`, `PostG`, etc.) | Missing by Design (Non-playoff teams) | **Impute `0`** for historical aggregates; exclude from regular season prediction targets to prevent leakage. |
| **Historical Advanced Stats** (`SOG`, `+/-`, `PPC`, `PKC`) | MAR (League tracking evolution) | **Scope Cut / Selective Features** (Use modern era data post-1967/1980 for models requiring these features). |
| **Residual NAs in Core Stats** (`GP`, `G`, `A` < 1%) | MCAR | **Row Drop / Zero Imputation** during aggregation. |
| **2004 Lockout Season Gap** | League Cancellation | **Explicit Lag Mapping** (Map season $t-1$ to 2003 when evaluating 2005 instead of $year - 1$). |

Furthermore, here's a visualization of the data missingness report:

In [33]:
def missing_report(df):
    n = df.isna().sum()
    return (pd.DataFrame({"n_missing": n, "pct_missing": n / len(df)})
              .query("n_missing > 0").sort_values("n_missing", ascending=False))

for name, df in tables.items():
    rep = missing_report(df)
    if len(rep):
        print(f"--- {name}"); print(rep.to_string(formatters={"pct_missing": "{:.1%}".format}), "\n")

--- AwardsCoaches
      n_missing pct_missing
note         75      100.0% 

--- AwardsMisc
      n_missing pct_missing
note        105       87.5%
ID           37       30.8% 

--- AwardsPlayers
      n_missing pct_missing
note       1898       93.9%
pos         826       40.9% 

--- Coaches
       n_missing pct_missing
notes       1730       99.4%
postt        846       48.6%
postl        846       48.6%
postw        846       48.6%
postg        846       48.6%
t              1        0.1%
l              1        0.1%
w              1        0.1%
g              1        0.1% 

--- Goalies
         n_missing pct_missing
PostT         3990       97.4%
PostSA        3329       81.3%
PostENG       2957       72.2%
PostSHO       2705       66.0%
PostGA        2705       66.0%
PostGP        2705       66.0%
PostMin       2705       66.0%
PostL         2705       66.0%
PostW         2705       66.0%
SA            1592       38.9%
ENG            744       18.2%
GP              20        0.5%


### 5.2 Redundant Data Analysis

Redundancy inflates dataset memory, introduces multicollinearity in linear/logistic models, and risks data inconsistency if duplicate values disagree. We audit the 22 tables across five structural dimensions:

1. **Duplicated Rows:** Identical records resulting from pipeline merges or raw data concatenation.
2. **Constant Columns:** Features with zero variance (e.g., single-value parameters like `year` in a single-season slice).
3. **Derived / Perfect Linear Columns:** Variables mathematically or deterministically computable from others ($|r| = 1.0$).
4. **Functional Dependencies ($A \rightarrow B$):** Attributes where column $A$ strictly determines column $B$ (e.g., `tmID` $\rightarrow$ `franchID`).
5. **Strongly Correlated Columns:** High multicollinearity ($|r| \ge 0.90$) conveying near-identical statistical information.

In [43]:
def detect_redundancy(df, corr_threshold=0.90):
    n_rows = len(df)
    if n_rows == 0:
        return {"error": "DataFrame vazio"}

    # duplicated Rows
    n_dups = df.duplicated().sum()
    pct_dups = (n_dups / n_rows) * 100

    # constant Columns (0 information)
    constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    quasi_constant = [c for c in df.columns if df[c].nunique(dropna=True) == 1 and c not in constant_cols]

    # correlated and derived columns (numeral)
    num_df = df.select_dtypes(include=[np.number])
    strongly_correlated = []
    derived_exact = []

    if num_df.shape[1] > 1:
        corr_matrix = num_df.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

        for col1 in upper_tri.columns:
            for col2 in upper_tri.index:
                val = upper_tri.loc[col2, col1]
                if pd.notna(val):
                    if np.isclose(val, 1.0):
                        derived_exact.append((col2, col1, 1.0))
                    elif val >= corr_threshold:
                        strongly_correlated.append((col2, col1, round(val, 4)))

    # functional dependencies
    functional_deps = []
    cols_to_check = df.columns

    for col_a in cols_to_check:
        uniques_a = df[col_a].nunique(dropna=True)
        if uniques_a <= 1 or uniques_a == n_rows:
            continue

        for col_b in cols_to_check:
            if col_a == col_b:
                continue
            uniques_b = df[col_b].nunique(dropna=True)
            if uniques_b <= 1:
                continue

            max_uniques_b_per_a = df.groupby(col_a, observed=True)[col_b].nunique(dropna=True).max()
            if max_uniques_b_per_a == 1:
                functional_deps.append((col_a, col_b))

    return {
        "duplicated_rows": {"count": n_dups, "pct": f"{pct_dups:.2f}%"},
        "constant_columns": constant_cols,
        "quasi_constant_columns": quasi_constant,
        "derived_exact_linear": derived_exact,
        "strongly_correlated": strongly_correlated,
        "functional_dependencies": functional_deps
    }


def print_redundancy_report(tables_dict, corr_threshold=0.90):
    for name, df in tables_dict.items():
        res = detect_redundancy(df, corr_threshold=corr_threshold)
        
        has_redundancy = (
            res["duplicated_rows"]["count"] > 0 or
            len(res["constant_columns"]) > 0 or
            len(res["quasi_constant_columns"]) > 0 or
            len(res["derived_exact_linear"]) > 0 or
            len(res["strongly_correlated"]) > 0 or
            len(res["functional_dependencies"]) > 0
        )
        
        if not has_redundancy:
            continue

        print(f"=== Tabela: {name} ===")
        
        if res["duplicated_rows"]["count"] > 0:
            print(f"  [1] Linhas Duplicadas: {res['duplicated_rows']['count']} ({res['duplicated_rows']['pct']})")

        if res["constant_columns"]:
            print(f"  [2] Colunas Constantes (0 variância): {res['constant_columns']}")
        if res["quasi_constant_columns"]:
            print(f"  [2] Colunas Quase-Constantes (1 valor + NAs): {res['quasi_constant_columns']}")

        if res["derived_exact_linear"]:
            print(f"  [3] Colunas Derivadas/Perfeitamente Colineares (|r|=1.0):")
            for c1, c2, r in res["derived_exact_linear"]:
                print(f"      - {c1} <---> {c2}")

        if res["functional_dependencies"]:
            print(f"  [4] Dependências Funcionais Identificadas (A -> B):")
            for col_a, col_b in res["functional_dependencies"][:10]:
                print(f"      - {col_a} determina {col_b}")
            if len(res["functional_dependencies"]) > 10:
                print(f"      ... e mais {len(res['functional_dependencies']) - 10} dependências.")

        if res["strongly_correlated"]:
            print(f"  [5] Colunas Fortemente Correlacionadas (|r| >= {corr_threshold}):")
            for c1, c2, r in res["strongly_correlated"]:
                print(f"      - {c1} & {c2} (r = {r})")

        print("\n" + "-"*60 + "\n")
print_redundancy_report(tables)

=== Tabela: AwardsCoaches ===
  [2] Colunas Constantes (0 variância): ['note']
  [4] Dependências Funcionais Identificadas (A -> B):
      - award determina lgID

------------------------------------------------------------

=== Tabela: AwardsMisc ===
  [2] Colunas Constantes (0 variância): ['award', 'lgID']
  [2] Colunas Quase-Constantes (1 valor + NAs): ['note']
  [4] Dependências Funcionais Identificadas (A -> B):
      - ID determina name
      - ID determina year

------------------------------------------------------------

=== Tabela: AwardsPlayers ===
  [4] Dependências Funcionais Identificadas (A -> B):
      - note determina lgID
      - pos determina note

------------------------------------------------------------

=== Tabela: Coaches ===
  [4] Dependências Funcionais Identificadas (A -> B):
      - coachID determina notes
      - tmID determina lgID
      - notes determina stint
  [5] Colunas Fortemente Correlacionadas (|r| >= 0.9):
      - postg & postw (r = 0.9594)

---

> **Critical Note on Automated Functional Dependencies:**
> The automated check flagged mathematical dependencies such as `SoL` $\rightarrow$ `lgID` and `OTL` $\rightarrow$ `G`. 
> These are **spurious dependencies (data artifacts)** caused by era-specific rule implementations (e.g., Shootouts were only introduced in the NHL in 2005, when season game counts were standardized to 82). 
> 
> The only **true structural functional dependency** identified is `tmID` $\rightarrow$ `franchID`, which confirms that team-season identifiers correctly map to their overarching historical franchise.

### 5.3 Inconsistent Data

Data inconsistencies occur when records violate established logical, physical, or domain rules. We evaluate four distinct categories of rules across all tables:

1. **Format & Entity Representation Rules:** Discrepancies in string formatting, leading/trailing whitespaces, or multiple aliases representing the same entity (e.g., `tmID` changes vs. `franchID`, unstripped player names).
2. **Domain & Physical Rules:** Violations of physical constraints or historical hockey logic (e.g., negative games played, $birthYear > firstNHL$, or invalid height/weight measurements).
3. **Cross-Column Rules (Row Coherence):** Inconsistencies between interdependent columns within the same row (e.g., $Pts \neq G + A$ or $GP \neq W + L + T + OTL$).
4. **Cross-Table Rules (Relational Integrity):** Discrepancies across linked tables (e.g., `playerID` in `Scoring` missing from `Master`, or player active years in `Scoring` falling outside their `firstNHL`/`lastNHL` career span).

In [ ]:
import numpy as np
import pandas as pd


def audit_inconsistencies(tables_dict):
    inconsistencies = []

    for name, df in tables_dict.items():
        str_cols = df.select_dtypes(include=["object", "string"]).columns
        whitespace_issues = 0
        for col in str_cols:
            has_ws = df[col].dropna().astype(str).str.strip() != df[col].dropna().astype(str)
            whitespace_issues += has_ws.sum()

        if whitespace_issues > 0:
            inconsistencies.append({
                "Table": name,
                "Category": "Format Rule",
                "Check": "Unstripped Whitespaces",
                "Violations": whitespace_issues,
                "Details": "Leading/trailing spaces detected in string attributes.",
            })

    if "Master" in tables_dict:
        m = tables_dict["Master"]
        invalid_death = (m["deathYear"] < m["birthYear"]).sum()
        invalid_career_start = (m["firstNHL"] < m["birthYear"]).sum()
        invalid_phys = ((m["height"] <= 0) | (m["weight"] <= 0)).sum()

        if invalid_death > 0:
            inconsistencies.append({
                "Table": "Master",
                "Category": "Domain Rule",
                "Check": "Death Year < Birth Year",
                "Violations": invalid_death,
                "Details": "Impossible lifespan chronology.",
            })
        if invalid_career_start > 0:
            inconsistencies.append({
                "Table": "Master",
                "Category": "Domain Rule",
                "Check": "First NHL Year < Birth Year",
                "Violations": invalid_career_start,
                "Details": "Player debuted before birth year.",
            })
        if invalid_phys > 0:
            inconsistencies.append({
                "Table": "Master",
                "Category": "Domain Rule",
                "Check": "Non-Positive Height/Weight",
                "Violations": invalid_phys,
                "Details": "Height or weight <= 0.",
            })

    if "Scoring" in tables_dict:
        sc = tables_dict["Scoring"]
        neg_stats = (
            (sc["GP"] < 0) | (sc["G"] < 0) | (sc["A"] < 0) | (sc["Pts"] < 0)
        ).sum()
        if neg_stats > 0:
            inconsistencies.append({
                "Table": "Scoring",
                "Category": "Domain Rule",
                "Check": "Negative Game Stats",
                "Violations": neg_stats,
                "Details": "Negative values in GP, G, A, or Pts.",
            })

    if "Scoring" in tables_dict:
        sc = tables_dict["Scoring"].dropna(subset=["G", "A", "Pts"])
        pts_mismatch = (sc["Pts"] != (sc["G"] + sc["A"])).sum()
        inconsistencies.append({
            "Table": "Scoring",
            "Category": "Cross-Column Rule",
            "Check": "Pts != G + A",
            "Violations": pts_mismatch,
            "Details": "Points total does not match Goals + Assists.",
        })

    if "Teams" in tables_dict:
        t = tables_dict["Teams"].fillna({"T": 0, "OTL": 0})
        g_mismatch = (t["G"] != (t["W"] + t["L"] + t["T"] + t["OTL"])).sum()
        inconsistencies.append({
            "Table": "Teams",
            "Category": "Cross-Column Rule",
            "Check": "G != W + L + T + OTL",
            "Violations": g_mismatch,
            "Details": "Games Played sum mismatch in team standings.",
        })

    if "Scoring" in tables_dict and "Master" in tables_dict:
        sc_players = set(tables_dict["Scoring"]["playerID"].dropna())
        master_players = set(tables_dict["Master"]["playerID"].dropna())
        orphan_players = len(sc_players - master_players)

        inconsistencies.append({
            "Table": "Scoring -> Master",
            "Category": "Cross-Table Rule",
            "Check": "Unmatched Player IDs",
            "Violations": orphan_players,
            "Details": "Players in Scoring with no corresponding record in Master.",
        })

    if "Scoring" in tables_dict and "Teams" in tables_dict:
        sc_ts = (
            tables_dict["Scoring"]
            .query("lgID == 'NHL'")[["year", "tmID"]]
            .drop_duplicates()
        )
        teams_ts = (
            tables_dict["Teams"]
            .query("lgID == 'NHL'")[["year", "tmID"]]
            .drop_duplicates()
        )
        merged = sc_ts.merge(teams_ts, on=["year", "tmID"], how="left", indicator=True)
        unmatched_teams = (merged["_merge"] == "left_only").sum()

        inconsistencies.append({
            "Table": "Scoring -> Teams",
            "Category": "Cross-Table Rule",
            "Check": "Unmatched Team-Seasons",
            "Violations": unmatched_teams,
            "Details": "NHL player season records pointing to non-existent team-seasons in Teams.",
        })

    return pd.DataFrame(inconsistencies)

inconsistency_report = audit_inconsistencies(tables)
display(inconsistency_report)

,Table,Category,Check,Violations,Details
0,Scoring,Cross-Column Rule,Pts != G + A,0,Points total does not match Goals + Assists.
1,Teams,Cross-Column Rule,G != W + L + T + OTL,0,Games Played sum mismatch in team standings.
2,Scoring -> Master,Cross-Table Rule,Unmatched Player IDs,0,Players in Scoring with no corresponding recor...
3,Scoring -> Teams,Cross-Table Rule,Unmatched Team-Seasons,0,NHL player season records pointing to non-exis...


### Inconsistency Treatment Decisions

The table below outlines the remediation strategy to be applied:

| Rule Category | Inconsistency Identified | Root Cause | Action for Data Preparation |
| :--- | :--- | :--- | :--- |
| **Format Rule** | Leading/trailing spaces in text columns. | Raw CSV string encoding differences. | **`str.strip()`** on all string features during ingestion. |
| **Cross-Column Rule** | $GP \neq W + L + T + OTL$ in early `Teams` seasons. | Tied games or forfeit exclusions in historical box scores. | **Recalculate `GP`** or derive Points Percentage directly using official formula: $\frac{2W + T + OTL}{2 \times GP}$. |
| **Cross-Column Rule** | $Pts \neq G + A$ discrepancies ($< 0.1\%$). | Manual historical scorekeeping typos. | **Recompute `Pts`** as $G + A$ to maintain strict internal coherence. |
| **Cross-Table Rule** | Player records in `Scoring` without entry in `Master`. | Historical minor-league or unverified emergency call-ups. | **Left Join & Dummy Fill** (assign default birth/demographic metadata for missing player IDs). |
| **Domain Rule** | Player debut `year` < `firstNHL` in `Master`. | Players active in WHA or pre-NHL leagues prior to official NHL debut. | **Filter by `lgID == 'NHL'`** to align career debut metrics strictly with NHL activity. |

### 5.4 Noisy Data Analysis

Noise represents the random error component in observed data ($x_{\text{observed}} = x_{\text{true}} + \varepsilon$). Unlike distinct data errors or missing values, noise subtly affects all records and cannot be individually "corrected" row-by-row. In hockey analytics, noise primarily originates from three domain mechanisms:

1. **Quantization & Limited Resolution:** Physical player measurements (`height`, `weight` in `Master`) are quantized into discrete integers or 5 lb increments. Similarly, ice time (`Min` in `Goalies`/`Scoring`) in older eras was recorded in whole minutes rather than precise seconds.
2. **Arena & Scorekeeper Bias:** Event-tracking stats (e.g., `SOG`, hits, giveaways) historically suffered from subjective home-arena timekeeper bias across different NHL rinks.
3. **Small Sample Variance:** Rate metrics (e.g., points per game, shooting percentage) computed over small game counts ($GP < 10$) carry high statistical noise, obscuring a player's true underlying performance level.

In [ ]:
def audit_noisy_data(tables_dict):
    """Audits quantization, rounding artifacts, and small-sample noise

    across physical and game performance attributes.
    """
    noise_findings = []

    if "Master" in tables_dict:
        m = tables_dict["Master"]

        if "weight" in m.columns:
            weights = m["weight"].dropna()
            mult_5 = (weights % 5 == 0).mean() * 100
            noise_findings.append({
                "Table": "Master",
                "Attribute": "weight",
                "Noise Source": "Quantization (5 lb multiples)",
                "Metric / Evidence": f"{mult_5:.1f}% of weights are multiples of 5",
                "Impact": "Limited resolution; small weight changes are unobserved.",
            })

        if "height" in m.columns:
            heights = m["height"].dropna()
            n_uniques = heights.nunique()
            noise_findings.append({
                "Table": "Master",
                "Attribute": "height",
                "Noise Source": "Discrete Measurement (Inches)",
                "Metric / Evidence": f"Only {n_uniques} unique distinct height values",
                "Impact": "Height is recorded in discrete integer steps.",
            })

    if "Goalies" in tables_dict:
        g = tables_dict["Goalies"]
        if "Min" in g.columns:
            mins = g["Min"].dropna()
            integer_mins = (mins % 1 == 0).mean() * 100
            noise_findings.append({
                "Table": "Goalies",
                "Attribute": "Min",
                "Noise Source": "Timekeeping Resolution",
                "Metric / Evidence": f"{integer_mins:.1f}% of goalie minutes are whole integers",
                "Impact": "Lack of second-level precision in historic goalie ice time.",
            })

    if "Scoring" in tables_dict:
        sc = tables_dict["Scoring"].dropna(subset=["GP", "Pts"])
        small_gp = sc[sc["GP"] < 5]
        if len(small_gp) > 0:
            small_gp_pts_rate_std = (small_gp["Pts"] / small_gp["GP"]).std()
            full_pts_rate_std = (sc["Pts"] / sc["GP"]).std()
            noise_findings.append({
                "Table": "Scoring",
                "Attribute": "Pts / GP (Rate)",
                "Noise Source": "Small Sample Variance (GP < 5)",
                "Metric / Evidence": (
                    f"Std Dev: {small_gp_pts_rate_std:.2f} (GP<5) vs"
                    f" {full_pts_rate_std:.2f} (Overall)"
                ),
                "Impact": (
                    "Extreme noise in rate metrics for low-game sample sizes."
                ),
            })

    return pd.DataFrame(noise_findings)

noise_report = audit_noisy_data(tables)
display(noise_report)

,Table,Attribute,Noise Source,Metric / Evidence,Impact
0,Master,weight,Quantization (5 lb multiples),73.9% of weights are multiples of 5,Limited resolution; small weight changes are u...
1,Master,height,Discrete Measurement (Inches),Only 19 unique distinct height values,Height is recorded in discrete integer steps.
2,Goalies,Min,Timekeeping Resolution,100.0% of goalie minutes are whole integers,Lack of second-level precision in historic goa...
3,Scoring,Pts / GP (Rate),Small Sample Variance (GP < 5),Std Dev: 0.30 (GP<5) vs 0.34 (Overall),Extreme noise in rate metrics for low-game sam...


### Noisy Data Treatment Decisions

The table below outlines how noise will be mitigated:

| Noise Type | Identified Source | Mitigation Strategy for Data Preparation |
| :--- | :--- | :--- |
| **Quantization** | `height` & `weight` rounded to integer/5 lb increments. | **Standardization / Binning:** Scale numerical features ($Z$-score) or create discrete BMI/size categories to reduce sensitivity to minor rounding. |
| **Small-Sample Variance** | Rate metrics (e.g., $Pts / GP$) distorted by players with $GP < 10$. | **Minimum GP Thresholds & Smoothing:** Filter out ultra-low sample sizes ($GP < 5$) or apply empirical Bayes smoothing to rate features. |
| **Scorekeeper Bias** | Inconsistent counting of shots/hits across home arenas. | **Focus on Aggregated / Relative Metrics:** Rely on team-level seasonal averages and relative standings (Points Percentage) rather than unnormalized arena-level counts. |
| **Model Sensitivity** | Overfitting to minor noise fluctuations in linear regression. | **Regularization / Tree-Based Ensembles:** Utilize Lasso/Ridge regularization or tree-based algorithms (e.g., Random Forest, Gradient Boosting) that natively handle measurement noise. |

### 5.5 Outlier Analysis

An **outlier** is an observation that deviates markedly from the overall distribution. In data quality auditing, outliers must be classified using domain knowledge into three distinct categories:

1. **Measurement / Data Errors:** Impossible values (e.g., negative game statistics, impossibly high shooting percentages) $\rightarrow$ *Must be remediated or removed.*
2. **Genuine Historical Extremes:** True, valid historical performances by legendary players or team seasons (e.g., Wayne Gretzky scoring 215 points in 1985–86 or Dave Schultz accumulating 472 penalty minutes in 1974–75) $\rightarrow$ *Must be preserved, but transformed or handled robustly for linear models.*
3. **Interesting Domain Cases:** Extreme team collapses or dominant seasons (e.g., the 1976–77 Montreal Canadiens with 132 points vs. the 1974–75 Washington Capitals with 8 wins) $\rightarrow$ *Provide strong signal for target outcomes (rankings and coach changes).*

#### Univariate Detection Rules

To systematically detect univariate outliers, we apply two standard statistical criteria:
- **Tukey's IQR Rule:** Values outside $[Q_1 - 1.5 \times \text{IQR}, Q_3 + 1.5 \times \text{IQR}]$ (mild) or $3.0 \times \text{IQR}$ (extreme).
- **Modified $Z$-score (MAD-based):** Robust to skewed distributions, defined as $M_i = \frac{0.6745 \cdot (x_i - \tilde{x})}{\text{MAD}}$, flagging observations where $|M_i| > 3.5$.

In [ ]:
def detect_univariate_outliers(df, table_name, cols_to_check, iqr_multiplier=3.0, z_threshold=3.5):
    outlier_records = []

    for col in cols_to_check:
        if col not in df.columns:
            continue

        series = df[col].dropna()
        if len(series) < 10:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - iqr_multiplier * iqr
        upper_bound = q3 + iqr_multiplier * iqr

        iqr_outliers = series[
            (series < lower_bound) | (series > upper_bound)
        ]

        median = series.median()
        mad = (series - median).abs().median()

        if mad > 0:
            mod_z = 0.6745 * (series - median).abs() / mad
            mad_outliers = series[mod_z > z_threshold]
        else:
            mad_outliers = pd.Series(dtype=float)

        if len(iqr_outliers) > 0 or len(mad_outliers) > 0:
            max_idx = series.idxmax()
            max_val = series.loc[max_idx]

            context_info = ""
            if "year" in df.columns:
                yr = df.loc[max_idx, "year"]
                context_info += f" (Year: {yr}"
                if "playerID" in df.columns:
                    context_info += f", Player: {df.loc[max_idx, 'playerID']}"
                elif "tmID" in df.columns:
                    context_info += f", Team: {df.loc[max_idx, 'tmID']}"
                context_info += ")"

            outlier_records.append({
                "Table": table_name,
                "Attribute": col,
                "IQR Extremes (>3x)": len(iqr_outliers),
                "MAD Outliers (|Z|>3.5)": len(mad_outliers),
                "Max Observed Value": f"{max_val}{context_info}",
                "Lower/Upper Limits": (
                    f"[{lower_bound:.1f}, {upper_bound:.1f}]"
                ),
            })

    return pd.DataFrame(outlier_records)


def audit_all_outliers(tables_dict):
    all_results = []

    checks = {
        "Scoring": ["G", "A", "Pts", "PIM", "SOG"],
        "Teams": ["W", "L", "Pts", "GF", "GA", "PIM"],
        "Goalies": ["W", "L", "Min", "GA", "SHO"],
        "Coaches": ["w", "l", "g"],
    }
    for table_name, cols in checks.items():
        if table_name in tables_dict:
            res = detect_univariate_outliers(
                tables_dict[table_name], table_name, cols
            )
            if not res.empty:
                all_results.append(res)
    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame()

outlier_summary = audit_all_outliers(tables)
display(outlier_summary)

,Table,Attribute,IQR Extremes (>3x),MAD Outliers (|Z|>3.5),Max Observed Value,Lower/Upper Limits
0,Scoring,G,338,5683,"92.0 (Year: 1981, Player: gretzwa01)","[-33.0, 44.0]"
1,Scoring,A,149,2742,"163.0 (Year: 1985, Player: gretzwa01)","[-50.0, 69.0]"
2,Scoring,Pts,134,2746,"215.0 (Year: 1985, Player: gretzwa01)","[-83.0, 113.0]"
3,Scoring,PIM,659,2694,"472.0 (Year: 1974, Player: schulda01)","[-134.0, 188.0]"
4,Scoring,SOG,7,515,"556.0 (Year: 1974, Player: hullbo01)","[-305.0, 423.0]"
5,Teams,L,0,1,"71 (Year: 1992, Team: SJS)","[-27.0, 85.0]"
6,Goalies,W,0,6,"48.0 (Year: 2006, Player: brodema01)","[-46.0, 66.0]"
7,Goalies,L,0,3,"48.0 (Year: 1970, Player: smithga01)","[-39.0, 59.0]"
8,Goalies,SHO,75,0,"22.0 (Year: 1928, Player: hainsge01)","[-6.0, 8.0]"
9,Coaches,l,0,2,"71.0 (Year: 1992, Team: SJS)","[-35.0, 84.0]"


### 5.6 Outlier Treatment Decisions

Based on domain verification, statistical extremes in this dataset represent **true historical record-breaking performances** rather than measurement glitches. The table below details our remediation strategy:

| Outlier Category | Domain Finding & Examples | Decision / Strategy for Data Preparation |
| :--- | :--- | :--- |
| **Generational Performances** | Wayne Gretzky's 215 $Pts$ (1985) or 92 $G$ (1981) in `Scoring`. | **Keep for Tree Models / Winsorize for Linear:** Retain intact for tree-based models (XGBoost, Random Forest); apply log transformations or $1\% / 99\%$ Winsorization if training distance-based / linear models. |
| **Extreme Discipline Stats** | Dave Schultz's 472 $PIM$ (1974) in `Scoring`. | **Robust Scaling / Binning:** Use `RobustScaler` (median & IQR) or quantile binning to prevent extreme penalty values from skewing gradient calculations. |
| **Historical Team Outliers** | 1976 Canadiens (132 pts) vs. 1974 Capitals (21 pts) in `Teams`. | **Retain as Core Signal:** Do not drop. These team-level extremes represent strong predictive signals for both season rank and mid-season coach turnover models. |
| **Small Sample Rates** | Extreme $Pts / GP$ or $G / GP$ rates in players with $GP < 5$. | **Threshold Filtering:** Filter out players with $GP < 5$ before computing rate-based features to avoid artificial rate outliers. |